### RAG Tracing with Phoenix

To run Phoenix server:
```
python -m phoenix.server.main serve
```

In [1]:
import json
import requests
from minsearch import AppendableIndex

from phoenix.otel import register
from opentelemetry import trace
from opentelemetry.trace import SpanKind, Status, StatusCode

from openai import OpenAI
from openinference.instrumentation.openai import OpenAIInstrumentor

In [2]:
endpoint='http://localhost:6006'
project_name = "lmz-rag-project"

tracer_provider = register(
    protocol="http/protobuf",
    project_name=project_name,
    endpoint="http://localhost:6006/v1/traces",
    auto_instrument=False
)

OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: lmz-rag-project
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [3]:
url_prefix = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/'
docs_url = url_prefix + 'search_evaluation/documents-with-ids.json'
documents = requests.get(docs_url).json()

In [4]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [12]:
def search(query, course="data-engineering-zoomcamp"):
    boost = {"question": 3.0, "section": 0.5}
        
    results = index.search(
        query=query,
        filter_dict={"course": course},
        boost_dict=boost,
        num_results=5,
        output_ids=True
    )
    return results

In [6]:
def build_prompt(query, search_results):
    prompt_template = """
You are the teaching assistant. Answer the QUESTION based on the CONTEXT given between ```.
Use only the facts from the CONTEXT to answer the QUESTION.

QUESTION: {question}
CONTEXT: ```{context}```
""".strip()

    context = ""
    for doc in search_results:
        context += f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    prompt = prompt_template.format(question=query, context=context)
    return prompt

In [7]:
client = OpenAI()

def llm(prompt, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [8]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [13]:
rag("how to install docker?")

'To install Docker on Ubuntu, you can use the following command:\n\n```\nsudo snap install docker\n```'

In [40]:
def rag_trace(query):
    tracer = tracer_provider.get_tracer(__name__)

    with tracer.start_as_current_span("LmzRag") as root_span:
        root_span.set_attribute("openinference.span.kind", "CHAIN")

        with tracer.start_as_current_span("Retriever") as span:
            span.set_attribute("openinference.instrumentation", "manual")
            span.set_attribute("openinference.span.kind", "RETRIEVER")
            span.set_attribute("retrieval.query", query)

            search_results = search(query)

            for d in search_results:
                span.add_event(
                    "retrieval.document",
                    {
                        "document.id": d["id"],
                        "document.content": d["text"],
                    },
                )

        prompt = build_prompt(query, search_results)

        with tracer.start_as_current_span("LLM") as llm_span:
            llm_span.set_attribute("openinference.span.kind", "LLM")
            llm_span.set_attribute("input.value", query)

            answer = llm(prompt)

            llm_span.set_attribute("output.value", answer)

        return answer


In [41]:
rag_trace("how to enroll the course?")

'To enroll in the course, you need to register before the course starts using the provided link.'